<div class="alert alert-block alert-success">
    <h1 align="center">COVID-19 Vaccination Prediction</h1>
    <h3 align="center">Machine Learning Project</h3>
</div>

<img src="https://www.eesc.europa.eu/sites/default/files/styles/large/public/images/shutterstock_1642888921.jpg?itok=P9-6YhGd" width=60%>

## 1. Importing the Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import warnings
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv('country_vaccinations.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nUnique Countries: {df['country'].nunique()}")

In [ ]:
iran_df = df[df['country'] == 'Iran'].copy()
iran_df = iran_df.sort_values('date').reset_index(drop=True)
print(f"Iran Dataset Shape: {iran_df.shape}")
iran_df.head()

## 3. EDA - Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.heatmap(iran_df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')

plt.subplot(1, 2, 2)
missing_pct = (iran_df.isnull().sum() / len(iran_df) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].plot(kind='barh')
plt.title('Missing Values Percentage')
plt.xlabel('Percentage (%)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(iran_df['date'], iran_df['daily_vaccinations'], linewidth=0.8, alpha=0.7, label='Daily')
rolling_mean = iran_df['daily_vaccinations'].rolling(window=7, min_periods=1).mean()
plt.plot(iran_df['date'], rolling_mean, linewidth=2, color='red', label='7-Day Rolling Mean')
plt.title('COVID-19 Daily Vaccinations in Iran')
plt.xlabel('Date')
plt.ylabel('Daily Vaccinations')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(iran_df['daily_vaccinations'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Daily Vaccinations')
axes[0].set_xlabel('Daily Vaccinations')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(iran_df['daily_vaccinations'].dropna()), bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Distribution of Log(Daily Vaccinations + 1)')
axes[1].set_xlabel('Log(Daily Vaccinations + 1)')

axes[2].boxplot(iran_df['daily_vaccinations'].dropna())
axes[2].set_title('Boxplot of Daily Vaccinations')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = iran_df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = iran_df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
iran_df = iran_df.copy()
iran_df['year'] = iran_df['date'].dt.year
iran_df['month'] = iran_df['date'].dt.month
iran_df['day'] = iran_df['date'].dt.day
iran_df['dayofweek'] = iran_df['date'].dt.dayofweek
iran_df['dayofyear'] = iran_df['date'].dt.dayofyear
iran_df['quarter'] = iran_df['date'].dt.quarter
iran_df['is_weekend'] = (iran_df['date'].dt.dayofweek >= 5).astype(int)

In [ ]:
target = 'daily_vaccinations'
for lag in [1, 3, 7, 14, 30]:
    iran_df[f'{target}_lag_{lag}'] = iran_df[target].shift(lag)

for window in [3, 7, 14, 30]:
    iran_df[f'{target}_rolling_{window}_mean'] = iran_df[target].rolling(window=window, min_periods=1).mean()
    iran_df[f'{target}_rolling_{window}_std'] = iran_df[target].rolling(window=window, min_periods=1).std()

In [ ]:
iran_df['daily_vaccinations_pct_change'] = iran_df[target].pct_change()
iran_df['daily_vaccinations_diff'] = iran_df[target].diff()
iran_df['month_sin'] = np.sin(2 * np.pi * iran_df['month'] / 12)
iran_df['month_cos'] = np.cos(2 * np.pi * iran_df['month'] / 12)

In [ ]:
iran_df = iran_df.dropna().reset_index(drop=True)
print(f"Shape after preprocessing: {iran_df.shape}")

In [ ]:
drop_cols = ['country', 'iso_code', 'vaccines', 'source_name', 'source_website', 'date', target]
feature_cols = [col for col in iran_df.columns if col not in drop_cols]

X = iran_df[feature_cols]
y = iran_df[target]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 5. Storytelling - Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if 'month' in X.columns:
    X.groupby('month')[target].mean().plot(kind='bar', ax=axes[0, 0], color='steelblue')
    axes[0, 0].set_title('Average Daily Vaccinations by Month')
    axes[0, 0].set_xlabel('Month')
    axes[0, 0].set_ylabel('Avg Daily Vaccinations')

if 'dayofweek' in X.columns:
    X.groupby('dayofweek')[target].mean().plot(kind='bar', ax=axes[0, 1], color='coral')
    axes[0, 1].set_title('Average Daily Vaccinations by Day of Week')
    axes[0, 1].set_xlabel('Day of Week')

if 'quarter' in X.columns:
    X.groupby('quarter')[target].mean().plot(kind='bar', ax=axes[1, 0], color='green')
    axes[1, 0].set_title('Average Daily Vaccinations by Quarter')
    axes[1, 0].set_xlabel('Quarter')

if 'is_weekend' in X.columns:
    X.groupby('is_weekend')[target].mean().plot(kind='bar', ax=axes[1, 1], color='purple')
    axes[1, 1].set_title('Average Daily Vaccinations: Weekday vs Weekend')
    axes[1, 1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
rf_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X, y)
importances = pd.Series(rf_temp.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20).plot(kind='barh', color='teal')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Prepare Data for Machine Learning

In [ ]:
top_features = importances.head(25).index.tolist()
X_selected = X[top_features]
print(f"Selected features ({len(top_features)}): {top_features}")

In [ ]:
split_idx = int(len(X_selected) * 0.8)

X_train, X_test = X_selected.iloc[:split_idx], X_selected.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 7. Train Your Model

In [ ]:
models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1
    )
}

trained_models = {}
cv_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    tscv = TimeSeriesSplit(n_splits=5)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=tscv,
                            scoring='neg_root_mean_squared_error', n_jobs=-1)
    rmse_scores = -scores
    cv_results[name] = {'mean_rmse': rmse_scores.mean(), 'std_rmse': rmse_scores.std()}
    print(f"  CV RMSE: {rmse_scores.mean():.2f} (+/- {rmse_scores.std():.2f})")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model

## 8. Test the Model and Show the Metrics

In [ ]:
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / np.maximum(np.abs(y_test), 1))) * 100
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape, 'predictions': y_pred}
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R2: {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, res) in enumerate(results.items()):
    axes[i].plot(y_test.values, label='Actual', linewidth=2)
    axes[i].plot(res['predictions'], label='Predicted', linewidth=2, linestyle='--')
    axes[i].set_title(f'{name}\nRMSE={res["RMSE"]:.2f}, R2={res["R2"]:.4f}')
    axes[i].set_xlabel('Time Index')
    axes[i].set_ylabel('Daily Vaccinations')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, res) in enumerate(results.items()):
    axes[i].scatter(y_test.values, res['predictions'], alpha=0.6, s=30)
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[i].set_title(f'{name}: Predicted vs Actual')
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
metrics_df = pd.DataFrame({name: {k: v for k, v in res.items() if k != 'predictions'}
                           for name, res in results.items()}).T
print("Model Comparison:")
print(metrics_df.to_string())

In [ ]:
metrics_df[['RMSE', 'MAE', 'R2', 'MAPE']].plot(kind='bar', figsize=(12, 6))
plt.title('Model Metrics Comparison')
plt.ylabel('Value')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = min(results, key=lambda x: results[x]['RMSE'])
best_model = trained_models[best_model_name]
print(f"Best Model: {best_model_name} (RMSE: {results[best_model_name]['RMSE']:.2f})")

## 9. Save Your Final Model

In [ ]:
joblib.dump(best_model, 'best_vaccination_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

model_info = {
    'model_name': best_model_name,
    'features': top_features,
    'rmse': results[best_model_name]['RMSE'],
    'r2': results[best_model_name]['R2']
}
pd.DataFrame([model_info]).to_csv('model_info.csv', index=False)

print("Model saved as 'best_vaccination_model.pkl'")
print("Scaler saved as 'scaler.pkl'")
print("Model info saved as 'model_info.csv'")

In [ ]:
loaded_model = joblib.load('best_vaccination_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
X_test_loaded = loaded_scaler.transform(X_test[top_features])
verify_pred = loaded_model.predict(X_test_loaded)
print(f"Verification RMSE: {np.sqrt(mean_squared_error(y_test, verify_pred)):.2f}")